# Modified Bessel Functions: `I_nu(x)` and `K_nu(x)` — Colab Test Suite

## Setup Instructions

**Step 1:** Change runtime to GPU: **Runtime > Change runtime type > T4 GPU > Save**

**Step 2:** Run Cell 1 — install the pre-built wheel (paste the download URL when prompted).

**Step 3:** Run Cells 2–14.

### How to get the wheel

1. Go to your fork: `https://github.com/shc443/pytorch/actions`
2. Click **"Build Wheel for Colab"** workflow > **Run workflow** > select `feature/bessel-arbitrary-order`
3. When done (~2-4 hrs), download `torch-colab-wheel` artifact (zip)
4. Extract the `.whl` file and upload it to Colab (drag into the file browser)

---

**Tests:** dtype coverage, precision vs SciPy (CPU+CUDA), broadcasting, scalar inputs,
edge cases, out= kwarg, CPU/CUDA parity, 1M element stress test.

In [ ]:
#@title Cell 1: Install pre-built wheel
# Upload the .whl file to Colab first (drag into file browser on the left),
# then run this cell.

!pip uninstall torch torchvision torchaudio -y -q 2>/dev/null

import glob
whl = glob.glob("/content/*.whl")
if not whl:
    raise FileNotFoundError(
        "No .whl file found in /content/. "
        "Upload the torch wheel to Colab's file browser first."
    )

print(f"Installing: {whl[0]}")
!pip install {whl[0]} -q
!pip install scipy -q

import torch
print(f"\nPyTorch {torch.__version__} installed successfully")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
#@title Cell 2: Import and verify
import torch
import scipy.special
import numpy as np
import time

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")

assert hasattr(torch.special, 'modified_bessel_i'), "modified_bessel_i not found!"
assert hasattr(torch.special, 'modified_bessel_k'), "modified_bessel_k not found!"
print("\ntorch.special.modified_bessel_i  FOUND")
print("torch.special.modified_bessel_k  FOUND")

In [ ]:
#@title Cell 3: Test helpers
def max_rel_error(ref, actual):
    """Max relative error, ignoring where both are zero or result is inf/nan."""
    ref = np.asarray(ref, dtype=np.float64)
    actual = np.asarray(actual, dtype=np.float64)
    mask = (ref != 0) & np.isfinite(ref) & np.isfinite(actual)
    if not mask.any():
        return 0.0
    return float(np.max(np.abs((ref[mask] - actual[mask]) / ref[mask])))


class Results:
    def __init__(self):
        self.passed = 0
        self.failed = 0
        self.skipped = 0
        self.failures = []

    def ok(self, name, condition):
        if condition:
            self.passed += 1
            print(f"  [PASS] {name}")
        else:
            self.failed += 1
            self.failures.append(name)
            print(f"  [FAIL] {name}")

    def skip(self, name, reason):
        self.skipped += 1
        print(f"  [SKIP] {name} ({reason})")

    def header(self, title):
        print(f"\n{'=' * 60}")
        print(f"  {title}")
        print(f"{'=' * 60}")

    def summary(self):
        total = self.passed + self.failed
        print(f"\n{'=' * 60}")
        print(f"  RESULTS: {self.passed}/{total} passed, {self.failed} failed, {self.skipped} skipped")
        print(f"{'=' * 60}")
        if self.failed == 0:
            print("  ALL TESTS PASSED")
        else:
            print(f"  FAILURES:")
            for f in self.failures:
                print(f"    - {f}")
        print()

R = Results()

In [ ]:
#@title Cell 4: Test 1 — Ops exist
R.header("TEST 1: Ops exist in torch.special")

R.ok("modified_bessel_i exists", hasattr(torch.special, 'modified_bessel_i'))
R.ok("modified_bessel_k exists", hasattr(torch.special, 'modified_bessel_k'))
R.ok("modified_bessel_i is callable", callable(getattr(torch.special, 'modified_bessel_i', None)))
R.ok("modified_bessel_k is callable", callable(getattr(torch.special, 'modified_bessel_k', None)))

In [ ]:
#@title Cell 5: Test 2 — Dtype coverage
R.header("TEST 2: Dtype coverage")

x = torch.tensor([1.0, 2.0, 3.0])
nu = torch.tensor([0.5, 1.0, 2.0])

for dtype in [torch.float32, torch.float64, torch.int32, torch.int64, torch.bool]:
    try:
        x_t = x.to(dtype)
        nu_t = nu.to(dtype)
        ri = torch.special.modified_bessel_i(x_t, nu_t)
        rk = torch.special.modified_bessel_k(x_t.float().clamp(min=0.1).to(dtype), nu_t)

        if dtype in (torch.int32, torch.int64, torch.bool):
            R.ok(f"I_nu dtype={dtype} -> float", ri.is_floating_point())
            R.ok(f"K_nu dtype={dtype} -> float", rk.is_floating_point())
        else:
            R.ok(f"I_nu dtype={dtype} preserves dtype", ri.dtype == dtype)
            R.ok(f"K_nu dtype={dtype} preserves dtype", rk.dtype == dtype)
    except Exception as e:
        R.ok(f"dtype={dtype}: {e}", False)

In [ ]:
#@title Cell 6: Test 3 — Precision vs SciPy (CPU)
R.header("TEST 3: Precision vs SciPy (CPU)")

np.random.seed(42)
x_np = np.random.uniform(0.1, 20.0, size=500).astype(np.float64)
nu_np = np.random.uniform(-10.0, 10.0, size=500).astype(np.float64)

for dtype, tol in [(torch.float32, 1e-3), (torch.float64, 1e-5)]:
    np_dtype = {torch.float32: np.float32, torch.float64: np.float64}[dtype]
    x_t = torch.tensor(x_np, dtype=dtype)
    nu_t = torch.tensor(nu_np, dtype=dtype)

    # I_nu: scipy.special.iv(nu, x)  -- note scipy uses (nu, x) arg order
    ref_i = scipy.special.iv(nu_np, x_np).astype(np_dtype)
    out_i = torch.special.modified_bessel_i(x_t, nu_t).numpy()
    err_i = max_rel_error(ref_i, out_i)
    R.ok(f"CPU I_nu {dtype} max_rel_err={err_i:.2e} (tol={tol})", err_i < tol)

    # K_nu: scipy.special.kv(nu, x)
    ref_k = scipy.special.kv(nu_np, x_np).astype(np_dtype)
    out_k = torch.special.modified_bessel_k(x_t, nu_t).numpy()
    err_k = max_rel_error(ref_k, out_k)
    R.ok(f"CPU K_nu {dtype} max_rel_err={err_k:.2e} (tol={tol})", err_k < tol)

In [ ]:
#@title Cell 7: Test 4 — Broadcasting
R.header("TEST 4: Broadcasting")

shapes = [
    ((5,), ()),            # vector x scalar
    ((), (5,)),            # scalar x vector
    ((5, 1), (5,)),        # broadcast along dim 1
    ((3, 4), ()),          # matrix x scalar
    ((3, 1, 4), (3, 4)),   # 3D broadcast
]

for s_x, s_nu in shapes:
    try:
        x_t = torch.rand(s_x, dtype=torch.float64) * 5.0 + 0.1
        nu_t = torch.rand(s_nu, dtype=torch.float64) * 4.0
        out_i = torch.special.modified_bessel_i(x_t, nu_t)
        out_k = torch.special.modified_bessel_k(x_t, nu_t)
        expected = torch.broadcast_shapes(s_x, s_nu)
        ok = (out_i.shape == expected) and (out_k.shape == expected)
        R.ok(f"broadcast {s_x} x {s_nu} -> {expected}", ok)
    except Exception as e:
        R.ok(f"broadcast {s_x} x {s_nu}: {e}", False)

In [ ]:
#@title Cell 8: Test 5 — Scalar input support
R.header("TEST 5: Scalar input support")

x_t = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float64)
nu_t = torch.tensor([0.5, 1.0, 2.0], dtype=torch.float64)

try:
    out = torch.special.modified_bessel_i(x_t, 1.5)
    R.ok("I_nu(tensor, py_scalar) shape", out.shape == (3,))
except Exception as e:
    R.ok(f"I_nu(tensor, py_scalar): {e}", False)

try:
    out = torch.special.modified_bessel_i(2.0, nu_t)
    R.ok("I_nu(py_scalar, tensor) shape", out.shape == (3,))
except Exception as e:
    R.ok(f"I_nu(py_scalar, tensor): {e}", False)

try:
    out = torch.special.modified_bessel_k(x_t, 1.5)
    R.ok("K_nu(tensor, py_scalar) shape", out.shape == (3,))
except Exception as e:
    R.ok(f"K_nu(tensor, py_scalar): {e}", False)

try:
    out = torch.special.modified_bessel_k(2.0, nu_t)
    R.ok("K_nu(py_scalar, tensor) shape", out.shape == (3,))
except Exception as e:
    R.ok(f"K_nu(py_scalar, tensor): {e}", False)

In [ ]:
#@title Cell 9: Test 6 — Edge cases
R.header("TEST 6: Edge cases")

# I_0(0) = 1
v = torch.special.modified_bessel_i(torch.tensor(0.0), torch.tensor(0.0)).item()
R.ok(f"I_0(0) = {v:.4f} (expect 1.0)", abs(v - 1.0) < 1e-6)

# I_nu(0) = 0 for positive integer nu
v = torch.special.modified_bessel_i(torch.tensor(0.0), torch.tensor(2.0)).item()
R.ok(f"I_2(0) = {v:.6f} (expect 0.0)", abs(v) < 1e-6)

# I_nu(0) = 0 for positive non-integer nu
v = torch.special.modified_bessel_i(torch.tensor(0.0), torch.tensor(0.5)).item()
R.ok(f"I_0.5(0) = {v:.6f} (expect 0.0)", abs(v) < 1e-6)

# I_{-nu}(x) = I_nu(x) for integer nu (DLMF 10.27.1)
x_t = torch.tensor([1.0, 2.0, 5.0], dtype=torch.float64)
i_pos = torch.special.modified_bessel_i(x_t, torch.tensor(2.0))
i_neg = torch.special.modified_bessel_i(x_t, torch.tensor(-2.0))
err = max_rel_error(i_pos.numpy(), i_neg.numpy())
R.ok(f"I_{{-2}}(x) == I_2(x), rel_err={err:.2e}", err < 1e-10)

# I_nu(x < 0) = NaN
v = torch.special.modified_bessel_i(torch.tensor(-1.0), torch.tensor(1.0)).item()
R.ok(f"I_1(-1) = NaN", np.isnan(v))

# K_nu(0) = inf
v = torch.special.modified_bessel_k(torch.tensor(0.0), torch.tensor(1.0)).item()
R.ok(f"K_1(0) = inf", v == float('inf') or v > 1e30)

# K_{-nu}(x) = K_nu(x) (DLMF 10.27.3)
x_t = torch.tensor([1.0, 2.0, 5.0], dtype=torch.float64)
k_pos = torch.special.modified_bessel_k(x_t, torch.tensor(2.5))
k_neg = torch.special.modified_bessel_k(x_t, torch.tensor(-2.5))
err = max_rel_error(k_pos.numpy(), k_neg.numpy())
R.ok(f"K_{{-2.5}}(x) == K_2.5(x), rel_err={err:.2e}", err < 1e-10)

# K_nu(x < 0) = NaN
v = torch.special.modified_bessel_k(torch.tensor(-1.0), torch.tensor(1.0)).item()
R.ok(f"K_1(-1) = NaN", np.isnan(v))

# Large order
try:
    v = torch.special.modified_bessel_i(torch.tensor(10.0, dtype=torch.float64), torch.tensor(50.0, dtype=torch.float64)).item()
    ref = scipy.special.iv(50.0, 10.0)
    err = abs(v - ref) / abs(ref) if ref != 0 else abs(v)
    R.ok(f"I_50(10) rel_err={err:.2e}", err < 1e-3)
except Exception as e:
    R.ok(f"I_50(10): {e}", False)

try:
    v = torch.special.modified_bessel_k(torch.tensor(10.0, dtype=torch.float64), torch.tensor(50.0, dtype=torch.float64)).item()
    ref = scipy.special.kv(50.0, 10.0)
    err = abs(v - ref) / abs(ref) if ref != 0 else abs(v)
    R.ok(f"K_50(10) rel_err={err:.2e}", err < 1e-3)
except Exception as e:
    R.ok(f"K_50(10): {e}", False)

# Fractional order
x_f = torch.tensor([1.0, 3.0, 7.0], dtype=torch.float64)
ref_i = scipy.special.iv(2.5, x_f.numpy())
out_i = torch.special.modified_bessel_i(x_f, torch.tensor(2.5)).numpy()
err = max_rel_error(ref_i, out_i)
R.ok(f"I_2.5(x) fractional order rel_err={err:.2e}", err < 1e-5)

ref_k = scipy.special.kv(2.5, x_f.numpy())
out_k = torch.special.modified_bessel_k(x_f, torch.tensor(2.5)).numpy()
err = max_rel_error(ref_k, out_k)
R.ok(f"K_2.5(x) fractional order rel_err={err:.2e}", err < 1e-5)

In [ ]:
#@title Cell 10: Test 7 — out= keyword argument
R.header("TEST 7: out= keyword argument")

x_t = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float64)
nu_t = torch.tensor([0.5, 1.0, 2.0], dtype=torch.float64)

out_buf = torch.empty(3, dtype=torch.float64)
ret = torch.special.modified_bessel_i(x_t, nu_t, out=out_buf)
R.ok("I_nu out= returns same tensor", ret.data_ptr() == out_buf.data_ptr())
R.ok("I_nu out= has correct values", torch.allclose(ret, torch.special.modified_bessel_i(x_t, nu_t)))

out_buf = torch.empty(3, dtype=torch.float64)
ret = torch.special.modified_bessel_k(x_t, nu_t, out=out_buf)
R.ok("K_nu out= returns same tensor", ret.data_ptr() == out_buf.data_ptr())
R.ok("K_nu out= has correct values", torch.allclose(ret, torch.special.modified_bessel_k(x_t, nu_t)))

In [ ]:
#@title Cell 11: Test 8 — Precision vs SciPy (CUDA)
R.header("TEST 8: Precision vs SciPy (CUDA)")

if not torch.cuda.is_available():
    R.skip("CUDA precision", "No CUDA available")
else:
    np.random.seed(123)
    x_np = np.random.uniform(0.1, 20.0, size=500).astype(np.float64)
    nu_np = np.random.uniform(-10.0, 10.0, size=500).astype(np.float64)

    for dtype, tol in [(torch.float32, 1e-3), (torch.float64, 1e-5)]:
        np_dtype = {torch.float32: np.float32, torch.float64: np.float64}[dtype]
        x_t = torch.tensor(x_np, dtype=dtype, device='cuda')
        nu_t = torch.tensor(nu_np, dtype=dtype, device='cuda')

        ref_i = scipy.special.iv(nu_np, x_np).astype(np_dtype)
        out_i = torch.special.modified_bessel_i(x_t, nu_t).cpu().numpy()
        err_i = max_rel_error(ref_i, out_i)
        R.ok(f"CUDA I_nu {dtype} max_rel_err={err_i:.2e} (tol={tol})", err_i < tol)

        ref_k = scipy.special.kv(nu_np, x_np).astype(np_dtype)
        out_k = torch.special.modified_bessel_k(x_t, nu_t).cpu().numpy()
        err_k = max_rel_error(ref_k, out_k)
        R.ok(f"CUDA K_nu {dtype} max_rel_err={err_k:.2e} (tol={tol})", err_k < tol)

In [ ]:
#@title Cell 12: Test 9 — CPU/CUDA parity
R.header("TEST 9: CPU/CUDA parity")

if not torch.cuda.is_available():
    R.skip("CPU/CUDA parity", "No CUDA available")
else:
    np.random.seed(456)
    x_np = np.random.uniform(0.1, 20.0, size=500).astype(np.float64)
    nu_np = np.random.uniform(-10.0, 10.0, size=500).astype(np.float64)

    x_cpu = torch.tensor(x_np, dtype=torch.float64)
    nu_cpu = torch.tensor(nu_np, dtype=torch.float64)
    x_gpu = x_cpu.cuda()
    nu_gpu = nu_cpu.cuda()

    cpu_i = torch.special.modified_bessel_i(x_cpu, nu_cpu)
    gpu_i = torch.special.modified_bessel_i(x_gpu, nu_gpu).cpu()
    err = max_rel_error(cpu_i.numpy(), gpu_i.numpy())
    R.ok(f"I_nu CPU/CUDA parity rel_err={err:.2e}", err < 1e-12)

    cpu_k = torch.special.modified_bessel_k(x_cpu, nu_cpu)
    gpu_k = torch.special.modified_bessel_k(x_gpu, nu_gpu).cpu()
    err = max_rel_error(cpu_k.numpy(), gpu_k.numpy())
    R.ok(f"K_nu CPU/CUDA parity rel_err={err:.2e}", err < 1e-10)

    x_cpu32 = x_cpu.float()
    nu_cpu32 = nu_cpu.float()
    cpu_i32 = torch.special.modified_bessel_i(x_cpu32, nu_cpu32)
    gpu_i32 = torch.special.modified_bessel_i(x_cpu32.cuda(), nu_cpu32.cuda()).cpu()
    err = max_rel_error(cpu_i32.numpy(), gpu_i32.numpy())
    R.ok(f"I_nu CPU/CUDA parity float32 rel_err={err:.2e}", err < 1e-6)

In [ ]:
#@title Cell 13: Test 10 — CUDA stress test (1M elements)
R.header("TEST 10: CUDA large tensor stress test")

if not torch.cuda.is_available():
    R.skip("CUDA stress test", "No CUDA available")
else:
    N = 1_000_000
    x_big = torch.rand(N, dtype=torch.float32, device='cuda') * 20.0 + 0.1
    nu_big = torch.rand(N, dtype=torch.float32, device='cuda') * 20.0 - 10.0

    torch.cuda.synchronize()
    t0 = time.perf_counter()
    out_i = torch.special.modified_bessel_i(x_big, nu_big)
    torch.cuda.synchronize()
    dt_i = time.perf_counter() - t0

    torch.cuda.synchronize()
    t0 = time.perf_counter()
    out_k = torch.special.modified_bessel_k(x_big, nu_big)
    torch.cuda.synchronize()
    dt_k = time.perf_counter() - t0

    R.ok(f"I_nu 1M elements no crash, {dt_i:.3f}s", out_i.shape == (N,))
    R.ok(f"K_nu 1M elements no crash, {dt_k:.3f}s", out_k.shape == (N,))
    R.ok("I_nu 1M no NaN (positive x, finite nu)", not torch.isnan(out_i).any().item())
    R.ok("K_nu 1M no NaN (x > 0.1)", not torch.isnan(out_k).any().item())
    R.ok("I_nu 1M all finite (bounded inputs)", torch.isfinite(out_i).all().item())

In [ ]:
#@title Cell 14: Summary
R.summary()